# 01 · gf180mcu PDK, ngspice and LUT extraction for novaad

This notebook prepares everything novaad needs to size transistors in the open **GlobalFoundries 180 nm MCU** process:

1. Install the gf180mcu PDK ([google/gf180mcu-pdk](https://github.com/google/gf180mcu-pdk)) with [**ciel**](https://github.com/fossi-foundation/ciel).
2. Install **ngspice**.
3. Run a SPICE testbench that sweeps a single `nfet_03v3` / `pfet_03v3` and writes the **look-up tables (LUTs)** novaad reads.

Notebooks `02_cs_amplifier_design.ipynb` and `03_fd_amplifier_nsga2.ipynb` use the LUTs written here.

## 1. Install the gf180mcu PDK with ciel

`ciel` is the PDK version manager from the FOSSi Foundation (successor of `volare`). It downloads prebuilt releases of the Google/GF open PDK and places them under a PDK root (default `~/.ciel`).

In [1]:
# Global PDK settings: edit these
PDK_FAMILY = "gf180mcu"
PDK_VARIANT = "gf180mcuD"   # 5LM, 11 kÅ top metal: the variant used by most open-source tapeouts
PDK_VERSION = None          # None -> latest release, or pin a commit hash from `ciel ls-remote`
PDK_ROOT = "~/.ciel"

In [2]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

PDK_ROOT = Path(PDK_ROOT).expanduser()

# install ciel into this environment if it isn't on PATH yet
if shutil.which("ciel") is None and not (Path(sys.executable).parent / "ciel").exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "ciel"], check=True)
CIEL = shutil.which("ciel") or str(Path(sys.executable).parent / "ciel")
print(subprocess.run([CIEL, "--version"], capture_output=True, text=True).stdout.splitlines()[0])

Ciel v3.2.0 ©2022-2025 Efabless Corporation and Contributors


In [3]:
versions = subprocess.run(
    [CIEL, "ls-remote", "--pdk-family", PDK_FAMILY], capture_output=True, text=True, check=True
).stdout.split()
version = PDK_VERSION or versions[0]
print(f"{len(versions)} {PDK_FAMILY} releases available, enabling {version}")

proc = subprocess.run(
    [CIEL, "enable", "--pdk-root", str(PDK_ROOT), "--pdk-family", PDK_FAMILY, version],
    capture_output=True, text=True, check=True,
)
print(f"enabled {PDK_FAMILY} {version} in {PDK_ROOT}")

MODELS = PDK_ROOT / PDK_VARIANT / "libs.tech" / "ngspice"
assert (MODELS / "sm141064.ngspice").is_file(), f"ngspice models not found in {MODELS}"
sorted(p.name for p in MODELS.iterdir())

94 gf180mcu releases available, enabling e7a6777d157a9d94c7ac757457b9e4309e9e9000
enabled gf180mcu e7a6777d157a9d94c7ac757457b9e4309e9e9000 in /Users/gfdasdias/.ciel


['design.ngspice',
 'design.spice',
 'sm141064.ngspice',
 'sm141064.spice',
 'sm141064_mim.ngspice',
 'sm141064_mim.spice',
 'smbb000149.ngspice',
 'smbb000149.spice']

The 3.3 V devices `nfet_03v3` / `pfet_03v3` live in `sm141064.ngspice`. They are binned BSIM4 (level 54) models selected by the `typical`, `ff`, `ss`, `fs` and `sf` corner libraries, so they are instantiated as plain MOSFETs: `M1 d g s b nfet_03v3 w=... l=...`.

## 2. Install ngspice

In [4]:
import platform

if shutil.which("ngspice") is None:
    if platform.system() == "Darwin":
        subprocess.run(["brew", "install", "ngspice"], check=True)
    else:
        raise SystemExit("Install ngspice with your package manager, e.g. `sudo apt install ngspice` (version >= 40)")
print(subprocess.run(["ngspice", "-v"], capture_output=True, text=True).stdout)

******
** ngspice-47 : Circuit level simulation program
** Compiled with KLU Direct Linear Solver
** The U. C. Berkeley CAD Group
** Copyright 1985-1994, Regents of the University of California.
** Copyright 2001-2026, The ngspice team.
** Please get your ngspice manual from https://ngspice.sourceforge.io/docs.html
** Please file your bug-reports at http://ngspice.sourceforge.net/bugrep.html
******



`notebooks/spice.py` is a ~50-line helper shared by all notebooks:
- `spice.header()` returns the model includes.
- `spice.run(netlist)` runs `ngspice -b` in a temp dir.
- `spice.read(file)` loads `wrdata` output into a DataFrame.

A quick operating-point smoke test:

In [5]:
os.environ["PDK_ROOT"], os.environ["PDK"] = str(PDK_ROOT), PDK_VARIANT
import spice

wd = spice.run(spice.header() + '''
* smoke test: one nfet_03v3 biased in saturation
VG g 0 1.2
VD d 0 1.65
M1 d g 0 0 nfet_03v3 w=10u l=0.28u
.control
op
print @m1[id] @m1[gm] @m1[gds] @m1[vth] @m1[vdsat]
.endc
.end
''')
print("\n".join(l for l in (wd / "tb.log").read_text().splitlines() if l.startswith("@m1")))

@m1[id] = 6.847422e-04
@m1[gm] = 1.703194e-03
@m1[gds] = 7.083843e-05
@m1[vth] = 6.229030e-01
@m1[vdsat] = 4.224802e-01


## 3. LUT extraction testbench

novaad sizes devices from tables of small-signal parameters over a grid of `vgs × vds × vsb × lch`, all characterized at one reference width `REF_WIDTH`. The columns it needs are:

| column | meaning | source |
|---|---|---|
| `vgs vds vsb lch` | grid axes | sweep |
| `ids gm gds` | drain current and conductances | BSIM4 op-point |
| `cgg cgs cgd` | gate capacitances (intrinsic + overlap) | `cgg+cgso+cgdo`, `-cgs+cgso`, `-cgd+cgdo` |
| `cdb csb` | junction capacitances | `capbd`, `capbs` |
| `gmoverid jd ft av ron fom_bw` | derived figures | `gm/ids`, `ids/W`, `gm/(2π·cgg)`, `gm/gds`, `1/gds`, `ft·av` |

ngspice reports off-diagonal BSIM4 capacitances as transcapacitances, `cij = -∂Qi/∂Vj`. That is why `cgs`/`cgd` are negated. `cgg` from BSIM4 is intrinsic only, so the overlap terms are added back.

PMOS is swept with negated voltages. novaad takes `abs()` of the whole LUT, so both types end up with positive values.

In [6]:
# Global LUT sweep settings: edit these
VDD = 3.3                                   # sweep range for |vgs| and |vds| (3.3 V devices)
REF_WIDTH = 10e-6                           # characterization width; novaad scales W from it
LCH = [0.28e-6, 0.5e-6, 1e-6, 2e-6, 4e-6]   # channel lengths (0.28 um is the 3.3 V minimum)
VSB = [0.0, 0.4, 0.8, 1.2]                  # source-bulk voltages
VGS_STEP = 0.05
VDS_STEP = 0.1
CORNER = "typical"
TEMP = 27
OUT_DIR = Path("luts")

In [7]:
DEVICES = {"nch": "nfet_03v3", "pch": "pfet_03v3"}
PARAMS = ["id", "gm", "gds", "cgg", "cgs", "cgd", "cgso", "cgdo", "capbd", "capbs"]


def testbench(model: str, lch: float, vsb: float, sign: int) -> str:
    # sign=+1 for nmos, -1 for pmos (source grounded, all voltages negated)
    vecs = " ".join(["v(g)", "v(d)"] + [f"@m1[{p}]" for p in PARAMS])
    return spice.header(CORNER, TEMP) + f'''
* {model} LUT sweep: L={lch}, VSB={vsb}
VG g 0 0
VD d 0 0
VB b 0 {-sign * vsb}
M1 d g 0 b {model} w={REF_WIDTH} l={lch}
.control
set wr_singlescale
set wr_vecnames
save {vecs}
dc VG 0 {sign * VDD} {sign * VGS_STEP} VD {sign * VDS_STEP} {sign * VDD} {sign * VDS_STEP}
wrdata lut.txt {vecs}
.endc
.end
'''


print(testbench("nfet_03v3", LCH[0], VSB[0], +1))

* novaad testbench
.include /Users/gfdasdias/.ciel/gf180mcuD/libs.tech/ngspice/design.ngspice
.lib /Users/gfdasdias/.ciel/gf180mcuD/libs.tech/ngspice/sm141064.ngspice typical
.temp 27
.option savecurrents

* nfet_03v3 LUT sweep: L=2.8e-07, VSB=0.0
VG g 0 0
VD d 0 0
VB b 0 -0.0
M1 d g 0 b nfet_03v3 w=1e-05 l=2.8e-07
.control
set wr_singlescale
set wr_vecnames
save v(g) v(d) @m1[id] @m1[gm] @m1[gds] @m1[cgg] @m1[cgs] @m1[cgd] @m1[cgso] @m1[cgdo] @m1[capbd] @m1[capbs]
dc VG 0 3.3 0.05 VD 0.1 3.3 0.1
wrdata lut.txt v(g) v(d) @m1[id] @m1[gm] @m1[gds] @m1[cgg] @m1[cgs] @m1[cgd] @m1[cgso] @m1[cgdo] @m1[capbd] @m1[capbs]
.endc
.end



The sweep starts at `|vds| = VDS_STEP` because at `vds = 0` the drain current is zero and `gm/id` would be infinite. Each `(lch, vsb)` pair is one ngspice run, and each run does a nested `vgs × vds` DC sweep.

In [8]:
from math import pi
from pandas import DataFrame, concat


def extract(dev: str) -> DataFrame:
    frames = []
    for lch in LCH:
        for vsb in VSB:
            wd = spice.run(testbench(DEVICES[dev], lch, vsb, +1 if dev == "nch" else -1))
            raw = spice.read(wd / "lut.txt")
            frames.append(DataFrame({
                "vgs": raw["v(g)"], "vds": raw["v(d)"], "vsb": vsb, "lch": lch,
                "ids": raw["m1.id"], "gm": raw["m1.gm"], "gds": raw["m1.gds"],
                "cgg": raw["m1.cgg"] + raw["m1.cgso"] + raw["m1.cgdo"],
                "cgs": -raw["m1.cgs"] + raw["m1.cgso"],
                "cgd": -raw["m1.cgd"] + raw["m1.cgdo"],
                "cdb": raw["m1.capbd"], "csb": raw["m1.capbs"],
            }))
    lut = concat(frames, ignore_index=True).abs()
    lut[["vgs", "vds"]] = lut[["vgs", "vds"]].round(6)   # exact grid values (the GUI needs them)
    lut = lut[lut["ids"] > 0].copy()
    lut["gmoverid"] = lut["gm"] / lut["ids"]
    lut["jd"] = lut["ids"] / REF_WIDTH
    lut["ft"] = lut["gm"] / (2 * pi * lut["cgg"])
    lut["av"] = lut["gm"] / lut["gds"]
    lut["ron"] = 1 / lut["gds"]
    lut["fom_bw"] = lut["ft"] * lut["av"]
    return lut


OUT_DIR.mkdir(exist_ok=True)
LUTS = {}
for dev in DEVICES:
    LUTS[dev] = extract(dev)
    LUTS[dev].to_csv(OUT_DIR / f"gf180_{dev}.csv", index=False)
    print(f"{dev}: {len(LUTS[dev])} rows -> {OUT_DIR / f'gf180_{dev}.csv'}")
LUTS["nch"].describe().T[["min", "max"]]

nch: 44220 rows -> luts/gf180_nch.csv


pch: 44220 rows -> luts/gf180_pch.csv


,min,max
vgs,0.000000e+00,3.300000e+00
vds,1.000000e-01,3.300000e+00
vsb,0.000000e+00,1.200000e+00
lch,2.800000e-07,4.000000e-06
ids,3.950585e-20,5.082803e-03
gm,1.217969e-18,2.088047e-03
gds,1.312152e-21,5.828873e-03
cgg,7.540352e-15,1.624947e-13
cgs,1.672680e-15,1.299937e-13
cgd,1.210398e-15,8.345727e-14


To use the LUTs from the novaad **CLI**, add them to your `pyproject.toml`:

In [9]:
print("[tool.novaad.nch]")
print(f'lut_path = "{(OUT_DIR / "gf180_nch.csv").resolve()}"')
print(f"ref_width = {REF_WIDTH}\n")
print("[tool.novaad.pch]")
print(f'lut_path = "{(OUT_DIR / "gf180_pch.csv").resolve()}"')
print(f"ref_width = {REF_WIDTH}")

[tool.novaad.nch]
lut_path = "/Users/gfdasdias/Documents/GF_Software/novaad/notebooks/luts/gf180_nch.csv"
ref_width = 1e-05

[tool.novaad.pch]
lut_path = "/Users/gfdasdias/Documents/GF_Software/novaad/notebooks/luts/gf180_pch.csv"
ref_width = 1e-05


## Sanity check: load the LUTs with novaad

These are the classic gm/id design charts at `|vds| = VDD/2` and `vsb = 0`, one curve per channel length. NMOS gm/id should approach roughly 25–30 V⁻¹ in weak inversion, and short-channel `ft` should reach the GHz range.

In [10]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from bokeh.io import output_notebook, show
from bokeh.layouts import gridplot
from bokeh.palettes import Category10
from bokeh.plotting import figure
from novaad import Device

output_notebook(hide_banner=True)

CHARTS = [  # (x, y, x_axis_type, y_axis_type)
    ("vgs", "gmoverid", "linear", "linear"),
    ("gmoverid", "jd", "linear", "log"),
    ("gmoverid", "ft", "linear", "log"),
    ("gmoverid", "av", "linear", "log"),
]

rows = []
for dev in DEVICES:
    device = Device(OUT_DIR / f"gf180_{dev}.csv", device_type=dev, ref_width=REF_WIDTH)
    lut = device.lut
    vds = lut["vds"].unique()[abs(lut["vds"].unique() - VDD / 2).argmin()]
    row = []
    for x, y, xt, yt in CHARTS:
        p = figure(title=f"{dev}: {y} vs {x}", x_axis_label=x, y_axis_label=y,
                   x_axis_type=xt, y_axis_type=yt, width=330, height=260)
        for lch, color in zip(LCH, Category10[10]):
            d = lut.query("abs(vds - @vds) < 1e-6 and vsb == 0 and abs(lch - @lch) < 1e-9").sort_values("vgs")
            p.line(d[x], d[y], color=color, legend_label=f"{lch * 1e6:g} um")
        p.legend.label_text_font_size = "7pt"
        p.legend.location = "top_left" if y != "gmoverid" else "top_right"
        row.append(p)
    rows.append(row)
show(gridplot(rows))